# Fase 1 · Paso 3: piloto de separación bilateral y lateralidad

Esta libreta prepara diez vistas seudonimizadas para revisar la línea de separación y la correspondencia derecha/izquierda sin consultar el desenlace. No procesa la cohorte completa, no recorta la articulación, no entrena y no abre la prueba reservada.

## 1. Montar Drive y localizar la raíz privada

In [ ]:
import os
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')
private_root = userdata.get('KNEE_DATA_ROOT') or os.environ.get('KNEE_DATA_ROOT')
if not private_root:
    raise RuntimeError('Configura el secreto KNEE_DATA_ROOT antes de continuar.')
os.environ['KNEE_DATA_ROOT'] = private_root
private_root_path = Path(private_root)
if not private_root_path.is_dir():
    raise FileNotFoundError('La raíz privada configurada no está disponible.')
pilot_audit_path = private_root_path / 'outputs' / 'auditorias' / 'fase_1' / 'paso_1_piloto' / 'auditoria_piloto_privada.json'
if not pilot_audit_path.is_file():
    raise FileNotFoundError('No se encontró la auditoría cerrada del piloto.')
print('Drive montado, raíz privada localizada y piloto cerrado disponible.')

## 2. Obtener una revisión identificable del repositorio

In [ ]:
import subprocess

REPO_URL = 'https://github.com/AlonsoYess/Knee.git'
BRANCH = 'codex/fase-0-base-reproducible'
REPO_DIR = Path('/content/Knee_fase1_paso3')

if REPO_DIR.exists():
    if not (REPO_DIR / '.git').is_dir():
        raise RuntimeError('La ruta de trabajo existe y no es un clon Git; usa un entorno nuevo.')
    dirty = subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout
    if dirty:
        raise RuntimeError('El clon temporal contiene cambios. Restablece el entorno de ejecución.')
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'merge', '--ff-only', f'origin/{BRANCH}'], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
commit = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout.strip()
print('Revisión preparada:', commit)

## 3. Instalar y comprobar el código

In [ ]:
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=REPO_DIR, check=True)
if subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout:
    raise RuntimeError('Las comprobaciones modificaron el repositorio temporal.')
print('Código y pruebas automáticas preparados.')

## 4. Preparar el piloto ciego

El comando usa exclusivamente el acta privada ya cerrada del piloto y los diez paquetes canónicos.

In [ ]:
config_path = REPO_DIR / 'configs' / 'bilateral_separation.example.json'
subprocess.run([sys.executable, '-m', 'knee.bilateral_separation', '--config', str(config_path)], cwd=REPO_DIR, check=True)

## 5. Mostrar las diez vistas seudonimizadas para revisión

La línea roja es la separación propuesta. La revisión debe comprobar que divide ambas rodillas sin cortar anatomía y que los marcadores o la anatomía respaldan la correspondencia mostrada. Una confianza baja obliga a revisión; no equivale por sí sola a exclusión.

In [ ]:
import json
from IPython.display import display
from PIL import Image

output_dir = private_root_path / 'outputs' / 'auditorias' / 'fase_1' / 'paso_3_separacion_bilateral'
summary_path = output_dir / 'resumen_separacion_publico.json'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
if summary.get('status') != 'ready_for_blinded_review' or summary.get('processed_unique_studies') != 10:
    raise RuntimeError('La preparación técnica del piloto no quedó completa.')
for preview_path in sorted((output_dir / 'previews_ciegas').glob('case_*_separacion.png')):
    print(preview_path.stem.split('_separacion')[0])
    with Image.open(preview_path) as preview:
        display(preview.copy())
print('Revisión pendiente de documentar en:', output_dir / 'revision_visual_ciega.csv')

## 6. Registrar la preparación

Este registro no cierra el paso 3. Después de revisar las diez vistas, se completará el archivo `revision_visual_ciega.csv` y se decidirá si los parámetros se congelan o necesitan ajuste dentro del piloto.

In [ ]:
from datetime import datetime, timezone

record = {
    'phase': 1,
    'step': 3,
    'operation': 'prepare_blinded_bilateral_pilot',
    'executed_at_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': commit,
    'status': summary['status'],
    'processed_unique_studies': summary['processed_unique_studies'],
    'visual_review_status': summary['visual_review_status'],
    'mass_processing_executed': False,
    'training_executed': False,
    'reserved_test_opened': False,
}
(output_dir / 'registro_preparacion.json').write_text(json.dumps(record, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
print('Fase 1, paso 3 preparado para revisión visual ciega. El paso aún no está cerrado.', record)